# PySpark: daily sales per store and product family

The same transformation as the dbt model `fct_sales_daily_store_family`, written in PySpark on Databricks.
The goal: build it as a small medallion pipeline, then prove that it gives the same totals as the PostgreSQL version.

**Layers:**
- **Bronze:** the raw tables in `workspace.raw`, loaded straight from Kaggle into Databricks.
- **Silver:** cleaned items, with tidy family names.
- **Gold:** one row per date, store and product family.

## 1. Bronze: check the raw tables

The raw tables were loaded by notebook `01_load_raw_favorita`. Before any logic, check their column types and the load quality.

In [ ]:
# Bronze: the raw tables loaded by notebook 01_load_raw_favorita (straight from Kaggle into Databricks)
train = spark.table("workspace.raw.train")
items = spark.table("workspace.raw.items")

print("train columns:")
train.printSchema()

print("items columns:")
items.printSchema()

train columns:
root
 |-- id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- store_nbr: short (nullable = true)
 |-- item_nbr: integer (nullable = true)
 |-- unit_sales: float (nullable = true)
 |-- onpromotion: boolean (nullable = true)

items columns:
root
 |-- item_nbr: integer (nullable = true)
 |-- family: string (nullable = true)
 |-- class: integer (nullable = true)
 |-- perishable: integer (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [ ]:
from pyspark.sql import functions as F

# Bronze check: read_files puts values that did not fit their column into _rescued_data.
# A clean load has no such rows.
rescued_rows = items.filter(F.col("_rescued_data").isNotNull()).count()

print("items rows: ", items.count())
print("items rows with rescued data: ", rescued_rows)

items rows:  4100
items rows with rescued data:  0


## 2. Silver: clean the items table

Tidy the family names with the same rule as the dbt model `stg_items`: commas, slashes and spaces become `_`, then lowercase. Example: `BREAD/BAKERY` becomes `bread_bakery`.

`train` needs no silver step: its column types were fixed when it was loaded.

In [ ]:
# Silver: the same rule as the dbt model stg_items (DI-19)
#   commas, slashes and spaces -> _ , then lowercase
#   e.g. 'BREAD/BAKERY' -> 'bread_bakery'
items_silver = items.select(
    F.col("item_nbr"),
    F.lower(F.regexp_replace(F.col("family"), "[,/ ]", "_")).alias("family"),
    F.col("class").alias("item_class"),
    (F.col("perishable") == 1).alias("is_perishable"),
)

# A separate schema, so this never collides with the dbt models built on Databricks
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.pyspark")
items_silver.write.mode("overwrite").saveAsTable("workspace.pyspark.silver_items")

saved = spark.table("workspace.pyspark.silver_items")
print("silver_items rows: ", saved.count())
print("distinct families: ", saved.select("family").distinct().count())
saved.orderBy("family").show(5)

silver_items rows:  4100
distinct families:  33
+--------+----------+----------+-------------+
|item_nbr|    family|item_class|is_perishable|
+--------+----------+----------+-------------+
|  270522|automotive|      6806|        false|
|  312317|automotive|      6810|        false|
|  172995|automotive|      6824|        false|
|  121964|automotive|      6810|        false|
|  461432|automotive|      6810|        false|
+--------+----------+----------+-------------+
only showing top 5 rows


### Findings

- **Bronze types:** `train` has the explicit types set at load time. The `items` types were guessed by `read_files`.
- **Load quality:** 0 of 4,100 `items` rows have rescued data.
- **Silver items:** 4,100 rows and 33 families, the same as the dbt model `stg_items` in PostgreSQL.

## 3. Gold: daily sales per store and product family

The same logic as the dbt model `fct_sales_daily_store_family`:
- Each sales row gets its item's family (left join to silver items).
- The rows are grouped by date, store and family.
- `unit_sales` is summed as an exact decimal, because the source column is a 4-byte float.
- `items_on_promotion` stays NULL when promotions were not recorded (before 2014-04-01).

In [ ]:
# Gold reads the saved silver table, not the variable: each layer reads the layer before it
silver_items = spark.table("workspace.pyspark.silver_items")

# each sales row gets its item's family (left join: no sales row is ever dropped)
sales_with_family = train.join(silver_items, on="item_nbr", how="left")

# the grain: one row per date, store and family
grouped = sales_with_family.groupBy("date", "store_nbr", "family")

gold = grouped.agg(
    # exact decimal sum: a 4-byte float loses small amounts on big totals
    F.sum(F.col("unit_sales").cast("decimal(28,6)")).alias("unit_sales"),
    # number of items with a sales row (a sale or a return)
    F.count("*").alias("items_recorded"),
    # true = 1, false = 0, NULL stays NULL: NULL before 2014-04-01 (DI-13)
    F.sum(F.col("onpromotion").cast("int")).alias("items_on_promotion"),
)

gold.write.mode("overwrite").saveAsTable("workspace.pyspark.gold_sales_daily_store_family")

gold_saved = spark.table("workspace.pyspark.gold_sales_daily_store_family")
print("gold rows: ", gold_saved.count())
print("gold rows with no family: ", gold_saved.filter(F.col("family").isNull()).count())

gold rows:  2061903
gold rows with no family:  0


### Findings

- **Gold rows:** 2,061,903, the same row count as the dbt model `fct_sales_daily_store_family` in PostgreSQL.
- **Join quality:** 0 rows with no family, so every sales row found its item.

## 4. Reconciliation: gold against bronze

The gold table must add up to the bronze `train` table exactly: the same number of rows and the same total units.

In [ ]:
# Bronze side: count the rows and sum the units with the same exact cast as gold
bronze_rows = train.count()
bronze_units = train.select(F.sum(F.col("unit_sales").cast("decimal(28,6)"))).first()[0]

# Gold side: items_recorded counts bronze rows, so its total must equal the bronze row count
gold_rows = gold_saved.select(F.sum("items_recorded")).first()[0]
gold_units = gold_saved.select(F.sum("unit_sales")).first()[0]

print("bronze rows: ", bronze_rows)
print("gold items_recorded total: ", gold_rows)
print("rows match: ", bronze_rows == gold_rows)
print()
print("bronze total units: ", bronze_units)
print("gold total units: ", gold_units)
print("units match: ", bronze_units == gold_units)

bronze rows:  125497040
gold items_recorded total:  125497040
rows match:  True

bronze total units:  1073610268.790475
gold total units:  1073610268.790475
units match:  True


### Findings

- **Rows:** 125,497,040 in bronze and in gold. Exact match.
- **Units:** 1,073,610,268.790475 in bronze and in gold. Exact match.

## 5. Cross-platform check: Databricks against PostgreSQL

The same totals per year, from both platforms. Row counts must match exactly. Unit totals may differ in the decimals, because each engine converts a 4-byte float to a decimal in its own way.

In [ ]:
# Totals per year from the Databricks gold table
gold_with_year = gold_saved.withColumn("year", F.year("date"))
grouped_by_year = gold_with_year.groupBy("year")

totals_by_year = grouped_by_year.agg(
    F.sum("items_recorded").alias("row_count"),
    F.sum("unit_sales").alias("total_units"),
)

totals_by_year.orderBy("year").show()

+----+---------+----------------+
|year|row_count|     total_units|
+----+---------+----------------+
|2013| 16322662|140417999.929633|
|2014| 22271602|209474121.583630|
|2015| 27864644|240867830.647378|
|2016| 35229871|288633375.254584|
|2017| 23808261|194216941.375250|
+----+---------+----------------+



### Findings

PostgreSQL totals from the dbt model `fct_sales_daily_store_family`, by the same query:

| Year | Rows (both) | Units: Databricks | Units: PostgreSQL | Difference |
|---|---|---|---|---|
| 2013 | 16,322,662 | 140,417,999.929633 | 140,417,999.918 | +0.012 |
| 2014 | 22,271,602 | 209,474,121.583630 | 209,474,121.537 | +0.047 |
| 2015 | 27,864,644 | 240,867,830.647378 | 240,867,830.647 | +0.0004 |
| 2016 | 35,229,871 | 288,633,375.254584 | 288,633,375.205 | +0.050 |
| 2017 | 23,808,261 | 194,216,941.375250 | 194,216,941.389 | -0.014 |
| **Total** | **125,497,040** | **1,073,610,268.790475** | **1,073,610,268.696** | **+0.094** |

- **Rows:** identical in every year. Both engines apply the same logic.
- **Units:** the totals differ by at most 0.05 units a year, and by 0.094 units out of 1.07 billion overall.
- **Cause:** the two engines convert each 4-byte float to a decimal differently. PostgreSQL rounds each value to about 6 significant digits. Spark keeps more digits. So the difference comes from the source type, not from the pipeline logic.